# Expanded-Dataset Autoencoder: Spatial Layout

Curated from the archived research notebook `new UCSD.ipynb`. Read the repository
README and `docs/limitations.md` before execution. All original files and
execution outputs were preserved separately.

The original workflow monitors arrays named `testing` during training.
Its displayed validation curves are not an independent final test. Training
is disabled until `ALLOW_TRAINING` is explicitly enabled.


In [ ]:
from pathlib import Path
import os
import sys

start = Path.cwd().resolve()
candidates = [start, *start.parents]
PROJECT_ROOT = next((p for p in candidates if (p / 'src/project_paths.py').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Start Jupyter from this repository or one of its notebook directories.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from project_paths import NotebookPaths
paths = NotebookPaths(PROJECT_ROOT, output_group='models/expanded_spatial_autoencoder')
input_path, input_glob, output_path = paths.input_path, paths.input_glob, paths.output_path
ALLOW_TRAINING = False  # Explicitly enable before running model training cells.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import scipy
from glob2 import glob


In [ ]:
file_path = 'new_ssh_training_data2D.nc'
ds = xr.open_dataset(input_path(file_path))
training = ds["ssh_training_data"].values
training_target =ds["ssh_training_target"].values
print(training.shape)


In [ ]:
file_path = 'new_ssh_testing_data2D.nc'
ds = xr.open_dataset(input_path(file_path))
testing = ds["ssh_testing_data"].values
testing_target =ds["ssh_testing_target"].values
print(testing.shape)


In [ ]:
import tensorflow as tf
import tensorflow.keras as tfkeras
from tensorflow.keras import regularizers
import tensorflow.keras.backend as K
from sklearn.model_selection import train_test_split
from matplotlib.pyplot import colorbar
from sklearn.utils import shuffle
from tensorflow.keras import regularizers
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix
from keras.layers import Input, Conv3D, Dropout, Dense, Flatten, Reshape, BatchNormalization, UpSampling3D, Conv3DTranspose
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping

# Split the data into training and testing sets
training = np.transpose(training, (3, 2, 0, 1))
training_target = np.transpose(training_target, (3, 2, 0, 1))
testing = np.transpose(testing, (3, 2, 0, 1))
testing_target = np.transpose(testing_target, (3, 2, 0, 1))
train, validation, train_target, validation_target = train_test_split(training, training_target, test_size=0.1, random_state=42)

# Check data types and shapes
print(f'x_train shape: {train.shape}, dtype: {train.dtype}')
print(f'y_train shape: {train_target.shape}, dtype: {train_target.dtype}')
print(f'x_test shape: {validation.shape}, dtype: {validation.dtype}')
print(f'y_test shape: {validation_target.shape}, dtype: {validation_target.dtype}')


In [ ]:
# Set the hyperparameters
learning_rate = 1e-4  # Adjust this value as needed
# Define a custom optimizer with gradient clipping
optimizer = Adam(learning_rate=learning_rate)#, clipnorm=1.0
Epoch=30000
#early_stop = tfkeras.callbacks.EarlyStopping(monitor='val_mse', patience=10000, verbose=0, mode='min',restore_best_weights='True')


In [ ]:
# Assuming train is already defined and has the shape (samples, time, x, y)
input_shape = (train.shape[1], train.shape[2], train.shape[3], 1)  # (time, x, y, channels)
input_img = Input(shape=input_shape)

# Encoder
x = Conv2D(16, (4, 8), padding='same')(input_img)  # Reduced filters
encoded = Conv2D(64, (4, 8), padding='same')(x)  # Reduced filters

# Decoder
x = Conv2DTranspose(64, (4, 8), padding='same')(encoded)  # Use Conv2DTranspose
x = Conv2DTranspose(16, (4, 8), padding='same')(x)  # Use Conv2DTranspose
decoded = Conv2D(1, (4, 8), activation='sigmoid', padding='same')(x)

# Combine encoder and decoder into an autoencoder model
autoencoder = Model(input_img, decoded)
autoencoder.compile(optimizer=optimizer, loss='mean_absolute_error', metrics=['mae'])

print(autoencoder.summary())


In [ ]:
if not ALLOW_TRAINING:
    raise RuntimeError('Set ALLOW_TRAINING=True after reviewing the epoch count and validation protocol.')

# Train the autoencoder
train_history = autoencoder.fit(train, train_target,
                epochs=Epoch,
                batch_size=32,
                shuffle=True,
                verbose=2,
                validation_data=(testing, testing_target))
                #callbacks=[early_stop]


In [ ]:
Training_History=np.zeros((4,Epoch))
Training_History[0,:] = np.array(train_history.history['loss'])
Training_History[1,:] = np.array(train_history.history['mae'])
Training_History[2,:] = np.array(train_history.history['val_loss'])
Training_History[3,:] = np.array(train_history.history['val_mae'])


In [ ]:
plt.plot(Training_History[1,:], color='blue', label='Training MAE')
plt.plot(Training_History[3,:], color='red', label='Monitored validation MAE')
plt.legend()
plt.xlabel('epochs')
plt.ylabel('MAE')
plt.show()


In [ ]:
# Save the entire model
autoencoder.save(output_path('autoencoder_model_1.h5'))


In [ ]:
denoised_ssha = autoencoder.predict(testing)
# Reshape the denoised images back to the original shape without the channel dimension
denoised_ssha = np.squeeze(denoised_ssha, axis=-1)
print(tf.keras.losses.MeanAbsoluteError()(testing , testing_target))
print(tf.keras.losses.MeanAbsoluteError()(denoised_ssha, testing_target))
